# UGLD Quickstart

This notebook shows the basics of **Uncertainty-Gated Lexical Decoding (UGLD)**.

We use GPT-2 (small, CPU-friendly) so no GPU is required.

In [1]:
%pip install ugld transformers --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList
from ugld import UGLD_Towards, UGLDTowardsConfig, UGLD_Against, UGLDAgainstConfig

model_name = "gpt2"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
print("Model ready.")

/home/mpapucci/ugld/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11302.73it/s]

Model ready.


## UGLD-t: condition *towards* a vocabulary

We define a small set of simple, common words as our **green vocabulary** and ask UGLD to
nudge the model towards using them when it is uncertain.

In [3]:
simple_words = [
    " simple", " easy", " basic", " clear",
    " small", " big", " light", " heavy",
    " fast", " slow", " old", " new",
    " good", " bad", " near", " far",
    " start", " end", " help", " use",
]

green_ids = list({id for w in simple_words for id in tok.encode(w, add_special_tokens=False)})
print(f"{len(green_ids)} green token ids")

20 green token ids


In [15]:
prompt = "Explain gravity in"
inputs = tok(prompt, return_tensors="pt")

# Baseline — no conditioning
with torch.no_grad():
    baseline = model.generate(**inputs, max_new_tokens=5, do_sample=False)
print("Baseline: ", tok.decode(baseline[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Baseline:  Explain gravity in the form of a circle


In [16]:
proc = LogitsProcessorList([
    UGLD_Towards(UGLDTowardsConfig(
        green_token_ids=green_ids,
        alpha_max=0.8,
        tau=5.0,
        s=0.3,
        prior="topk",
    ))
])

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=5, do_sample=False, logits_processor=proc)
print("UGLD-t:   ", tok.decode(out[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


UGLD-t:    Explain gravity in simple terms.




## UGLD-a: condition *against* a vocabulary

We define a set of technical words as our **red vocabulary** and ask UGLD to suppress them
when the model is uncertain.

In [17]:
technical_words = [
    " gravitational", " acceleration", " curvature", " spacetime",
    " relativity", " Newton", " Newtonian", " magnitude",
    " proportional", " inversely", " Einstein",
]

red_ids = list({id for w in technical_words for id in tok.encode(w, add_special_tokens=False)})
print(f"{len(red_ids)} red token ids")

14 red token ids


In [20]:
proc_a = LogitsProcessorList([
    UGLD_Against(UGLDAgainstConfig(
        red_token_ids=red_ids,
        lambda_max=4.0,
        tau=1.0,
        s=0.3,
        weights="fixed",
    ))
])

with torch.no_grad():
    out_a = model.generate(**inputs, max_new_tokens=5, do_sample=False, logits_processor=proc_a)
print("Baseline: ", tok.decode(baseline[0], skip_special_tokens=True))
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=5, do_sample=False, logits_processor=proc_a)
print("UGLD-a:   ", tok.decode(out_a[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Baseline:  Explain gravity in the form of a circle
UGLD-a:    Explain gravity in the form of a circle
